## Introduction
This notebook will perform extensive data analysis regarding bird bioacoustic monitoring. The data will be taken from the article "Automated identification of individual birds by song enables multi-year recapture from passive acoustic monitoring data", and it is located under resources/localized_events.csv. 

In [ ]:
# First, we must import the necessary packages and libraries
import pandas as pd
import ast
import matplotlib as mpl
import numpy as np
from pathlib import Path 
from plotnine import ( 
    ggplot, 
    aes, 
    geom_point,
    geom_bar,
    facet_wrap,
    labs,
    scale_fill_manual,
    theme_minimal
)

Then, we will need to load in the data as a dataframe or just fetch the binary if it exists

In [ ]:
path = Path("resources/localized_events.parquet")

if path.exists():
    print("Exists, so just load in binary file")
    all_loaded_events_df = pd.read_parquet("resources/localized_events.parquet")
else:
    print("Does not exist, so we create it")
    all_loaded_events_df = pd.read_csv("resources/localized_events.csv")
    all_loaded_events_df.to_parquet("resources/localized_events.parquet")


Now, we will take a subset of the column variables in the data spreadsheet.

In [ ]:
subst_loaded_events_df = all_loaded_events_df[
    [
    # Metadata
    'array',
    'data_split',
    'relative_position',
    'mic_distances_m',
    'start_timestamp',
    'duration',

    # Automated models
    'ovenbird_cluster_labels',
    'baseline_resnet18_cluster_labels',
    'hawkears_cluster_labels',
    'birdnet_cluster_labels',
    ]
]

### Exploratory Data Analysis
We will begin by exploring the distributions of the data to identify possible outliers, trends, and occurrences. 

In [ ]:
# add a new column individual so it does not conflict with python
subst_loaded_events_df['individual'] = all_loaded_events_df['class']

# Convert to datetime objects
subst_loaded_events_df['start_timestamp'] = pd.to_datetime(subst_loaded_events_df['start_timestamp'], utc=True)
subst_loaded_events_df['duration'] = pd.to_timedelta(subst_loaded_events_df['duration'], unit='sec')

In [ ]:
# Create a new column just for the end_timestamp
subst_loaded_events_df['end_timestamp'] = (subst_loaded_events_df['start_timestamp'] + subst_loaded_events_df['duration'])

In [ ]:
# Set an event key for all the rows
subst_loaded_events_df['event_key'] = range(len(subst_loaded_events_df))

In [ ]:
# .value_counts() creates a lookup table, (individual: counts) for all individuals
# .map() takes that lookup table and assigns it to those individuals using 'individual' as the key
# This new column 'count' is assigned with that counts for each individual
lookup_individual_counts = subst_loaded_events_df['individual'].value_counts()
subst_loaded_events_df['count'] = subst_loaded_events_df['individual'].map(
    lookup_individual_counts
)

In [ ]:
# Save to under resources
subst_loaded_events_df.to_parquet('resources/subset_loaded_events.parquet')
subst_loaded_events_df.to_csv('resources/subset_loaded_events.csv')

In [ ]:
# Creates a ggplot object 
(
    ggplot(subst_loaded_events_df, aes(x='factor(individual)', fill='count > 30'))
    + geom_bar()
    + scale_fill_manual(
        values={True: 'blue', False: 'red'}
    )
    + labs(
        title='Song Count per Individual',
        x='Individual',
        y='Count'
    )
)

Now, we want to plot the recordings according to the relative (x, y) coordinates, they will be labeled according to the individual they belong to & faceted by the sites they belong to.

In [ ]:
# Create the figure for this, using ast package to unpack the str
fig_df = pd.DataFrame()
fig_df['x'] = subst_loaded_events_df['relative_position'].apply(
    lambda p: ast.literal_eval(p)[0]
)
fig_df['y'] = subst_loaded_events_df['relative_position'].apply(
    lambda p: ast.literal_eval(p)[1]
)
fig_df['individual'] = subst_loaded_events_df['individual'].astype("category")

In [ ]:
(
    ggplot(fig_df, aes('x', 'y', fill = 'individual')) +
        geom_point() 
)

Now, we will facet grid them by the site.

In [ ]:
# Introduce the site column to enable facet gridding
fig_df['site'] = subst_loaded_events_df['array']

In [ ]:
(
    ggplot(fig_df, aes('x', 'y', fill = 'individual')) +
        geom_point() +
        facet_wrap('site') +
        labs(
        title='Ovenbird Song Positions by Array',
        x='Easting (m)',
        y='Northing (m)',
        color='Individual'    
        ) +
        theme_minimal()
)

### Bird Recordings Overlap
The overlaps of all the bird recordings will be found, and that information will be used to determine bird interactions.
- If two bird recordings overlap, that may or may not be an interaction (further analysis is required)
- If two bird recordings do not overlap, that is not an interaction

In [30]:
#subst_loaded_events_df.iloc[0]
len(subst_loaded_events_df)

717

In [ ]:
from pandas import DataFrame

# Create a Tuple that parses from a dataframe
def create_tuple_from_df(df: DataFrame):
    tuple_list = []
    for _, row in df[['event_key', 'start_timestamp', 'end_timestamp']].iterrows():
        tuple_list.append(tuple(row))
    return tuple_list

In [40]:
tuples_events = create_tuple_from_df(subst_loaded_events_df)
tuples_events

[(0,
  Timestamp('2016-06-25 14:10:54+0000', tz='UTC'),
  Timestamp('2016-06-25 14:10:57+0000', tz='UTC')),
 (1,
  Timestamp('2016-06-25 13:26:09+0000', tz='UTC'),
  Timestamp('2016-06-25 13:26:12+0000', tz='UTC')),
 (2,
  Timestamp('2016-06-22 11:43:15+0000', tz='UTC'),
  Timestamp('2016-06-22 11:43:18+0000', tz='UTC')),
 (3,
  Timestamp('2016-06-25 14:21:48+0000', tz='UTC'),
  Timestamp('2016-06-25 14:21:51+0000', tz='UTC')),
 (4,
  Timestamp('2016-06-25 14:20:15+0000', tz='UTC'),
  Timestamp('2016-06-25 14:20:18+0000', tz='UTC')),
 (5,
  Timestamp('2016-06-23 13:02:57+0000', tz='UTC'),
  Timestamp('2016-06-23 13:03:00+0000', tz='UTC')),
 (6,
  Timestamp('2016-06-25 14:11:24+0000', tz='UTC'),
  Timestamp('2016-06-25 14:11:27+0000', tz='UTC')),
 (7,
  Timestamp('2016-06-22 12:24:06+0000', tz='UTC'),
  Timestamp('2016-06-22 12:24:09+0000', tz='UTC')),
 (8,
  Timestamp('2016-06-25 13:13:51+0000', tz='UTC'),
  Timestamp('2016-06-25 13:13:54+0000', tz='UTC')),
 (9,
  Timestamp('2016-06-22

In [43]:
# Find the overlap from all the events
def overlap(events) -> list[tuple]:
    # We need to find all the ways the events overlap, then return a tuple of event ids 
    events_overlapped: list[tuple] = []
    events = sorted(events, key=lambda x: x[1]) # sort the events by the start, 

    for i, (key, start, end) in enumerate(events):
        for key2, start2, end2 in events[i+1:]:  # compare against all later events
            if start2 < end:  # overlap condition
                events_overlapped.append((key, key2))
            else:
                break  # since sorted, no further events can overlap

    return events_overlapped

In [45]:
# Test that sequence of events
overlap(tuples_events)

[(282, 285), (289, 290), (450, 452)]